In [1]:
import pickle

with open("/root/autodl-tmp/data/MOSI/aligned_50.pkl", "rb") as f:
    data = pickle.load(f)

print(type(data))



<class 'dict'>


In [2]:
print(data.keys())


dict_keys(['train', 'valid', 'test'])


In [4]:
first_key = list(data.keys())[0]
print(data[first_key].keys())


dict_keys(['raw_text', 'audio', 'vision', 'id', 'text', 'text_bert', 'annotations', 'classification_labels', 'regression_labels'])


In [6]:
data[first_key]['audio'].shape

(1284, 50, 5)

In [7]:
data[first_key]['raw_text'].shape

(1284,)

In [8]:
data[first_key]['vision'].shape

(1284, 50, 20)

In [ ]:
# 看一下视频是什么情况

import cv2
import os
import random

RAW_PATH = "/root/autodl-tmp/data/MOSI/Raw" 


def analyze_video(video_path):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"❌ 无法打开: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

    duration = total_frames / fps if fps > 0 else 0

    print("=" * 60)
    print("视频路径:", video_path)
    print(f"FPS: {fps:.2f}")
    print(f"总帧数: {int(total_frames)}")
    print(f"分辨率: {int(width)} x {int(height)}")
    print(f"时长(秒): {duration:.2f}")

    # 自动判断
    if fps < 10:
        print("⚠️ 低帧率版本 (很可能不是官方原始Raw)")
    elif fps < 25:
        print("⚠️ 非标准30fps版本")
    else:
        print("✅ 看起来像官方原始Raw版本")

    cap.release()


def random_sample_check():
    all_videos = []

    for folder in os.listdir(RAW_PATH):
        folder_path = os.path.join(RAW_PATH, folder)
        if not os.path.isdir(folder_path):
            continue

        for file in os.listdir(folder_path):
            if file.endswith(".mp4"):
                all_videos.append(os.path.join(folder_path, file))

    if not all_videos:
        print("❌ 没找到 mp4 文件")
        return

    sample_video = random.choice(all_videos)
    # sample_video = "/root/autodl-tmp/data/MOSI/Raw/9qR7uwkblbs/33.mp4"
    analyze_video(sample_video)


if __name__ == "__main__":
    random_sample_check()


视频路径: /root/autodl-tmp/data/MOSI/Raw/ZUXBRvtny7o/9.mp4
FPS: 30.00
总帧数: 106
分辨率: 320 x 240
时长(秒): 3.53
✅ 看起来像官方原始Raw版本


In [9]:
# 看一下视频的时长

import cv2
import os
import numpy as np
from tqdm import tqdm

RAW_PATH = "/root/autodl-tmp/data/MOSI/Raw"  # 改成你的路径


def collect_all_videos(root_path):
    all_videos = []
    for folder in os.listdir(root_path):
        folder_path = os.path.join(root_path, folder)
        if not os.path.isdir(folder_path):
            continue

        for file in os.listdir(folder_path):
            if file.endswith(".mp4"):
                all_videos.append(os.path.join(folder_path, file))

    return sorted(all_videos)


def analyze_all_videos():
    all_videos = collect_all_videos(RAW_PATH)

    if not all_videos:
        print("❌ 没找到 mp4 文件")
        return

    durations = []
    fps_list = []
    frame_counts = []

    print(f"共找到 {len(all_videos)} 个视频，开始统计...\n")

    for video_path in tqdm(all_videos):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

        duration = total_frames / fps if fps > 0 else 0

        durations.append(duration)
        fps_list.append(fps)
        frame_counts.append(total_frames)

        cap.release()

    durations = np.array(durations)
    fps_list = np.array(fps_list)
    frame_counts = np.array(frame_counts)

    print("\n" + "=" * 60)
    print("📊 视频统计结果")
    print("=" * 60)

    print(f"视频总数: {len(durations)}")

    print("\n🎬 时长统计（秒）")
    print(f"最短: {durations.min():.2f}")
    print(f"最长: {durations.max():.2f}")
    print(f"平均: {durations.mean():.2f}")
    print(f"中位数: {np.median(durations):.2f}")

    print("\n🎥 帧率统计")
    print(f"最小FPS: {fps_list.min():.2f}")
    print(f"最大FPS: {fps_list.max():.2f}")
    print(f"平均FPS: {fps_list.mean():.2f}")

    print("\n🖼️ 总帧数统计")
    print(f"最少帧: {frame_counts.min():.0f}")
    print(f"最多帧: {frame_counts.max():.0f}")
    print(f"平均帧数: {frame_counts.mean():.0f}")

    print("=" * 60)


if __name__ == "__main__":
    analyze_all_videos()


共找到 2199 个视频，开始统计...



100%|██████████| 2199/2199 [00:15<00:00, 139.98it/s]


📊 视频统计结果
视频总数: 2199

🎬 时长统计（秒）
最短: 0.23
最长: 52.50
平均: 4.29
中位数: 3.30

🎥 帧率统计
最小FPS: 30.00
最大FPS: 30.00
平均FPS: 30.00

🖼️ 总帧数统计
最少帧: 7
最多帧: 1575
平均帧数: 129


In [1]:
### 看看数据数量的情况
import pandas as pd

df = pd.read_csv("/root/autodl-tmp/data/CMU-MOSI/label.csv")

In [2]:
df

,video_id,clip_id,text,label,label_T,label_A,label_V,annotation,mode
0,03bSnISJMiM,11,A LOT OF SAD PARTS,-0.5,NaN,NaN,NaN,Neutral,train
1,03bSnISJMiM,10,THERE IS SAD PART,-1.2,NaN,NaN,NaN,Negative,train
2,03bSnISJMiM,13,AND ITS A REALLY FUNNY,1.8,NaN,NaN,NaN,Positive,train
3,03bSnISJMiM,12,BUT IT WAS REALLY REALLY AWESOME,2.2,NaN,NaN,NaN,Positive,train
4,03bSnISJMiM,1,ANYHOW IT WAS REALLY GOOD,2.4,NaN,NaN,NaN,Positive,train
...,...,...,...,...,...,...,...,...,...
2194,zhpQhgha_KU,30,BECAUSE THERE REALLY WASNT ALL THAT MUCH TO IT...,-2.0,NaN,NaN,NaN,Negative,test
2195,zhpQhgha_KU,35,UM SO IF YOU LIKE TO HEAR A UM LIKE MORE POSIT...,0.6,NaN,NaN,NaN,Positive,test
2196,zhpQhgha_KU,34,AND SHE REALLY ENJOYED THE FILM,1.2,NaN,NaN,NaN,Positive,test
2197,zhpQhgha_KU,33,IF YOU DO WANNA SEE SOMEBODY WHOS POSSIBLY CRI...,-0.4,NaN,NaN,NaN,Negative,test


In [6]:
df[df['mode'] == 'valid']

,video_id,clip_id,text,label,label_T,label_A,label_V,annotation,mode
1284,WKA5OygbEKI,20,I DEFINITELY WILL BUY IT,0.0,NaN,NaN,NaN,Neutral,valid
1285,WKA5OygbEKI,21,BUT UH IT WAS WORTH THE FREE MOVIE PASSES THAT...,1.6,NaN,NaN,NaN,Positive,valid
1286,WKA5OygbEKI,22,AND THANKS FOR LISTENING TO ME RAMBLE,0.4,NaN,NaN,NaN,Positive,valid
1287,WKA5OygbEKI,1,UH I REALLY ENJOYED DOING THEM FOR THE PHILLY ...,1.8,NaN,NaN,NaN,Positive,valid
1288,WKA5OygbEKI,3,UM I I LIKED A LOT OF THE MOVIE,1.2,NaN,NaN,NaN,Positive,valid
...,...,...,...,...,...,...,...,...,...
1508,c5xsKMxpXnc,4,WHAT I LIKED ABOUT THIS MOVIE WAS THE CONCEPT ...,2.0,NaN,NaN,NaN,Positive,valid
1509,c5xsKMxpXnc,7,THIS MOVIE IS TERRIBLE FOR THESE REASONS,-3.0,NaN,NaN,NaN,Negative,valid
1510,c5xsKMxpXnc,6,AND THEY LEFT A LOT OF PLOT HOLES,-2.0,NaN,NaN,NaN,Negative,valid
1511,c5xsKMxpXnc,9,YOU DONT FEEL THAT THESE TWO PEOPLE LOVE EACH ...,-1.4,NaN,NaN,NaN,Negative,valid


In [7]:
92 + 137

229

In [10]:
df[df['label'] >= 0]

,video_id,clip_id,text,label,label_T,label_A,label_V,annotation,mode
2,03bSnISJMiM,13,AND ITS A REALLY FUNNY,1.8,NaN,NaN,NaN,Positive,train
3,03bSnISJMiM,12,BUT IT WAS REALLY REALLY AWESOME,2.2,NaN,NaN,NaN,Positive,train
4,03bSnISJMiM,1,ANYHOW IT WAS REALLY GOOD,2.4,NaN,NaN,NaN,Positive,train
7,03bSnISJMiM,5,AND THEY SHOULDVE I GUESS,0.0,NaN,NaN,NaN,Neutral,train
9,03bSnISJMiM,7,AND XXX BUT BESIDES THAT IT WAS ALL OVER PRETT...,0.8,NaN,NaN,NaN,Positive,train
...,...,...,...,...,...,...,...,...,...
2190,zhpQhgha_KU,16,I THINK THAT CHARLOTTE AND MIRANDA HAVE SOME R...,1.8,NaN,NaN,NaN,Positive,test
2192,zhpQhgha_KU,18,ACTUALLY THE MOST UPBEAT CHARACTER IN THE WHOL...,0.6,NaN,NaN,NaN,Positive,test
2195,zhpQhgha_KU,35,UM SO IF YOU LIKE TO HEAR A UM LIKE MORE POSIT...,0.6,NaN,NaN,NaN,Positive,test
2196,zhpQhgha_KU,34,AND SHE REALLY ENJOYED THE FILM,1.2,NaN,NaN,NaN,Positive,test


In [11]:
df[df['label'] < 0]

,video_id,clip_id,text,label,label_T,label_A,label_V,annotation,mode
0,03bSnISJMiM,11,A LOT OF SAD PARTS,-0.50,NaN,NaN,NaN,Neutral,train
1,03bSnISJMiM,10,THERE IS SAD PART,-1.20,NaN,NaN,NaN,Negative,train
5,03bSnISJMiM,3,I MEAN THEY DID A LITTLE BIT OF IT,-1.00,NaN,NaN,NaN,Negative,train
6,03bSnISJMiM,2,THAY DID THEY DIDNT REALLY DO A WHOLE BUNCH OF...,-0.80,NaN,NaN,NaN,Negative,train
8,03bSnISJMiM,4,BUT NOT A WHOLE BUNCH,-1.75,NaN,NaN,NaN,Negative,train
...,...,...,...,...,...,...,...,...,...
2188,zhpQhgha_KU,14,BUT A LOT OF THE FOOTAGE WAS KIND OF UNNECESSARY,-1.50,NaN,NaN,NaN,Negative,test
2191,zhpQhgha_KU,19,I THINK THAT UM AND THAT EVERYBODY I DO FEEL R...,-0.80,NaN,NaN,NaN,Negative,test
2193,zhpQhgha_KU,31,UM LIKE I SAID IF YOU LIKE IT FEEL FREE TO GO ...,-1.00,NaN,NaN,NaN,Negative,test
2194,zhpQhgha_KU,30,BECAUSE THERE REALLY WASNT ALL THAT MUCH TO IT...,-2.00,NaN,NaN,NaN,Negative,test
